<a href="https://colab.research.google.com/github/zildoaranha/Desafio-Fiap/blob/notebooks/02_preprocessamento_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 — Pré-processamento

**Dimensão 4 da rúbrica — 15 pontos**

Nesta etapa foram realizadas a limpeza, transformação e preparação das bases `application_record` e `credit_record`.

O objetivo é construir uma base consistente para a modelagem de risco de crédito, documentando todas as decisões adotadas durante o pré-processamento.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

app = pd.read_csv("application_record.csv")
credit = pd.read_csv("credit_record.csv")

# 1. Análise Inicial

Antes dos tratamentos foi realizada uma avaliação inicial das duas bases, incluindo:

- dimensões;
- tipos de dados;
- registros duplicados;
- valores ausentes.

In [2]:
print("Application record")
print(app.shape)
display(app.head())
print(app.info())

print("\nCredit record")
print(credit.shape)
display(credit.head())
print(credit.info())

Application record
(438557, 18)


,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0
1,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,-21474,-1134,1,0,0,0,Security staff,2.0
3,5008808,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0
4,5008809,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 438557 entries, 0 to 438556
Data columns (total 18 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   ID                   438557 non-null  int64  
 1   CODE_GENDER          438557 non-null  object 
 2   FLAG_OWN_CAR         438557 non-null  object 
 3   FLAG_OWN_REALTY      438557 non-null  object 
 4   CNT_CHILDREN         438557 non-null  int64  
 5   AMT_INCOME_TOTAL     438557 non-null  float64
 6   NAME_INCOME_TYPE     438557 non-null  object 
 7   NAME_EDUCATION_TYPE  438557 non-null  object 
 8   NAME_FAMILY_STATUS   438557 non-null  object 
 9   NAME_HOUSING_TYPE    438557 non-null  object 
 10  DAYS_BIRTH           438557 non-null  int64  
 11  DAYS_EMPLOYED        438557 non-null  int64  
 12  FLAG_MOBIL           438557 non-null  int64  
 13  FLAG_WORK_PHONE      438557 non-null  int64  
 14  FLAG_PHONE           438557 non-null  int64  
 15  FLAG_EMAIL       

,ID,MONTHS_BALANCE,STATUS
0,5001711,0,X
1,5001711,-1,0
2,5001711,-2,0
3,5001711,-3,0
4,5001712,0,C


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 3 columns):
 #   Column          Non-Null Count    Dtype 
---  ------          --------------    ----- 
 0   ID              1048575 non-null  int64 
 1   MONTHS_BALANCE  1048575 non-null  int64 
 2   STATUS          1048575 non-null  object
dtypes: int64(2), object(1)
memory usage: 24.0+ MB
None


# 2. Dados Faltantes

Foi realizada a análise dos valores ausentes presentes nas bases.

A principal variável com ausência relevante foi `OCCUPATION_TYPE`.

In [3]:
print("Nulos em application_record")
print(app.isna().sum().sort_values(ascending=False))

print("\nNulos em credit_record")
print(credit.isna().sum().sort_values(ascending=False))

Nulos em application_record
OCCUPATION_TYPE        134203
ID                          0
CODE_GENDER                 0
FLAG_OWN_CAR                0
CNT_CHILDREN                0
FLAG_OWN_REALTY             0
NAME_INCOME_TYPE            0
NAME_EDUCATION_TYPE         0
NAME_FAMILY_STATUS          0
AMT_INCOME_TOTAL            0
NAME_HOUSING_TYPE           0
DAYS_BIRTH                  0
FLAG_MOBIL                  0
DAYS_EMPLOYED               0
FLAG_WORK_PHONE             0
FLAG_PHONE                  0
FLAG_EMAIL                  0
CNT_FAM_MEMBERS             0
dtype: int64

Nulos em credit_record
ID                0
MONTHS_BALANCE    0
STATUS            0
dtype: int64


## Decisão

Optou-se por preencher os valores ausentes de `OCCUPATION_TYPE` com a categoria **Unknown**.

### Justificativa

Os registros com ocupação ausente foram analisados em conjunto com outras variáveis e mantinham informações relevantes.

A exclusão dessas observações geraria perda significativa de informação sem benefício comprovado para a modelagem.

In [4]:
app["OCCUPATION_TYPE"] = (
    app["OCCUPATION_TYPE"]
    .fillna("Unknown")
)

# 3. Tratamento de Registros Duplicados

Foi identificada a existência de IDs duplicados na base cadastral.
`

In [5]:
print("Linhas duplicadas:", app.duplicated().sum())
print("IDs duplicados:", app["ID"].duplicated().sum())

ids_duplicados_app = (
    app[
        app["ID"].duplicated(keep=False)
    ]
    .sort_values("ID")
)

ids_duplicados_app.head()

Linhas duplicadas: 0
IDs duplicados: 47


,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS
425023,7022197,F,N,Y,0,450000.0,Commercial associate,Higher education,Separated,House / apartment,-19813,-1799,1,0,0,1,Unknown,1.0
426818,7022197,M,Y,Y,3,135000.0,Working,Secondary / secondary special,Married,House / apartment,-11945,-735,1,0,0,1,Laborers,5.0
431911,7022327,M,Y,Y,0,256500.0,Commercial associate,Higher education,Married,House / apartment,-21503,-1674,1,0,0,1,Core staff,2.0
431545,7022327,F,N,Y,0,135000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-14771,-5298,1,0,0,0,High skill tech staff,1.0
425486,7023108,M,Y,Y,1,67500.0,Working,Secondary / secondary special,Married,House / apartment,-15156,-1696,1,1,0,0,Core staff,3.0


## Decisão

Remover todos os registros associados a IDs duplicados.

### Justificativa

Os registros possuíam informações conflitantes para o mesmo cliente.

Como o ID será utilizado posteriormente como chave de integração com a variável alvo, manter esses registros poderia gerar inconsistências na base final.

In [6]:
ids_duplicados = app.loc[
    app["ID"].duplicated(keep=False),
    "ID"
].unique()

app = app[
    ~app["ID"].isin(ids_duplicados)
].copy()

print(app.shape)

(438463, 18)


# 4. Transformações de Variáveis

As colunas em dias foram convertidas para formatos mais interpretáveis.


In [7]:
app["AGE_YEARS"] = (
    -app["DAYS_BIRTH"] / 365
).round(1)

## Tratamento de DAYS_EMPLOYED

Foi identificado o valor especial 365243, que não representa um tempo de emprego real.

In [8]:
app["EMPLOYED_SPECIAL_VALUE"] = (
    app["DAYS_EMPLOYED"] == 365243
).astype(int)

app["DAYS_EMPLOYED_TREATED"] = (
    app["DAYS_EMPLOYED"]
    .replace(365243, pd.NA)
)

app["EMPLOYED_YEARS"] = (
    -pd.to_numeric(
        app["DAYS_EMPLOYED_TREATED"],
        errors="coerce"
    ) / 365
).round(1)

## Validação do tratamento

Após a transformação foi realizada uma verificação de consistência para garantir que todos os registros especiais foram identificados corretamente.

In [9]:
print(
    (app["DAYS_EMPLOYED"] == 365243).sum()
)

print(
    app["EMPLOYED_SPECIAL_VALUE"].sum()
)

print(
    app["EMPLOYED_YEARS"].isna().sum()
)

75314
75314
75314


# 5. Conversão de Variáveis Binárias

As variáveis de posse de carro e imóvel foram convertidas de Y/N para 1/0.

In [10]:
app["FLAG_OWN_CAR"] = np.where(
    app["FLAG_OWN_CAR"].isin(["Y",1,"1"]),
    1,
    0
)

app["FLAG_OWN_REALTY"] = np.where(
    app["FLAG_OWN_REALTY"].isin(["Y",1,"1"]),
    1,
    0
)

# 6. Análise da Base de Crédito

Antes da construção da variável alvo foi realizada a limpeza da base `credit_record`.


In [11]:
credit[
    credit["MONTHS_BALANCE"] == "-"
]

,ID,MONTHS_BALANCE,STATUS


In [12]:
credit = credit.loc[
    credit["MONTHS_BALANCE"] != "-"
].copy()

credit["MONTHS_BALANCE"] = (
    credit["MONTHS_BALANCE"]
    .astype(int)
)

# 7. Definição da Variável Alvo

A coluna STATUS representa a situação mensal de pagamento dos clientes.

In [13]:
credit["STATUS"].value_counts().sort_index()

,count
STATUS,
0,383120
1,11090
2,868
3,320
4,223
5,1693
C,442031
X,209230


## Critério adotado

Foram considerados eventos de inadimplência grave os status:

- 2 = atraso entre 60 e 89 dias
- 3 = atraso entre 90 e 119 dias
- 4 = atraso entre 120 e 149 dias
- 5 = atraso superior a 150 dias ou dívida baixada

Todos representam atrasos iguais ou superiores a 60 dias.

### Justificativa

A literatura de risco de crédito normalmente considera atrasos mais prolongados como um sinal mais forte de inadimplência. Dessa forma, optou-se por classificar como evento negativo clientes que apresentaram pelo menos um registro com atraso de 60 dias ou mais durante o período observado.

In [14]:
credit["BAD_MONTH"] = (
    credit["STATUS"]
    .isin(["2","3","4","5"])
    .astype(int)
)

In [15]:
target = (
    credit
    .groupby("ID")["BAD_MONTH"]
    .max()
    .reset_index()
)

target = target.rename(
    columns={
        "BAD_MONTH":"TARGET_BAD"
    }
)

## Justificativa

Utilizou-se o valor máximo por cliente.

Assim, qualquer cliente que tenha apresentado pelo menos um episódio de atraso grave é classificado como inadimplente.

# 8. Integração das Bases

In [16]:
df = app.merge(
    target,
    on="ID",
    how="inner"
)

print(df.shape)

(36457, 23)


## Decisão

Foi utilizado merge do tipo **inner**.

### Justificativa

Nem todos os clientes possuem simultaneamente cadastro e histórico de crédito.

O merge inner garante a presença das variáveis explicativas e da variável alvo.

# 9. Feature Engineering

Nesta etapa foram criadas variáveis derivadas com o objetivo de aumentar a interpretabilidade dos dados e facilitar o processo de modelagem.

Além da criação de novos atributos, também foram avaliados os valores extremos identificados durante a análise exploratória, definindo-se a estratégia de tratamento mais adequada para cada caso.

In [17]:
# Transformação logarítmica da renda

df["LOG_INCOME"] = np.log1p(
    df["AMT_INCOME_TOTAL"]
)

## Transformação da renda

### Justificativa

Durante a análise exploratória observou-se que a distribuição da variável AMT_INCOME_TOTAL apresenta forte assimetria à direita, com poucos clientes possuindo rendas significativamente superiores ao restante da população.

Em vez de remover registros ou limitar artificialmente os valores observados, optou-se por aplicar uma transformação logarítmica, criando a variável LOG_INCOME.

Essa abordagem reduz a influência dos valores extremos sobre a distribuição dos dados, preservando observações potencialmente relevantes para a modelagem e evitando perda de informação.

In [18]:
# Tratamento do valor especial de tempo de emprego

df["EMPLOYED_YEARS"] = (
    df["EMPLOYED_YEARS"]
    .fillna(-1)
)

## Tratamento do valor especial de emprego

### Justificativa

Durante o pré-processamento inicial foi identificado o valor especial 365243 na variável DAYS_EMPLOYED.

Após a criação da variável EMPLOYED_YEARS, os registros associados a esse valor permaneceram representados por valores ausentes.

Como esses casos já haviam sido identificados por meio da variável EMPLOYED_SPECIAL_VALUE, optou-se por substituir os valores ausentes por -1. Essa estratégia permite diferenciar claramente esses registros dos demais clientes sem perder informação.

In [19]:
# Verificação dos registros com valores extremos

df.sort_values(
    "AMT_INCOME_TOTAL",
    ascending=False
)[
    [
        "ID",
        "AMT_INCOME_TOTAL",
        "NAME_INCOME_TYPE",
        "OCCUPATION_TYPE"
    ]
].head(10)

df.sort_values(
    "CNT_CHILDREN",
    ascending=False
)[
    [
        "ID",
        "CNT_CHILDREN",
        "CNT_FAM_MEMBERS",
        "NAME_FAMILY_STATUS"
    ]
].head(10)

df.sort_values(
    "CNT_FAM_MEMBERS",
    ascending=False
)[
    [
        "ID",
        "CNT_CHILDREN",
        "CNT_FAM_MEMBERS",
        "NAME_FAMILY_STATUS"
    ]
].head(10)

,ID,CNT_CHILDREN,CNT_FAM_MEMBERS,NAME_FAMILY_STATUS
25412,5105054,19,20.0,Single / not married
14671,5061207,14,15.0,Separated
14672,5061210,14,15.0,Separated
14673,5061211,14,15.0,Separated
29827,5118331,7,9.0,Married
29826,5118330,7,9.0,Married
25860,5105613,5,7.0,Married
25857,5105610,5,7.0,Married
31857,5135540,5,7.0,Married
25856,5105609,5,7.0,Married


## Avaliação dos valores extremos

### Decisão

Os valores extremos identificados na análise exploratória foram mantidos na base.

### Justificativa

As análises realizadas mostraram que os registros com maiores valores de renda, quantidade de filhos e tamanho familiar apresentavam coerência entre as variáveis relacionadas.

Os clientes com maior quantidade de filhos também possuíam tamanhos familiares compatíveis, não sendo observadas evidências claras de erros de preenchimento ou inconsistências cadastrais.

No caso da renda anual, o impacto dos valores muito elevados foi reduzido por meio da transformação LOG_INCOME

# 10. Normalização / Padronização

## Decisão

Não realizar nesta etapa.

### Justificativa

A padronização será realizada posteriormente dentro do Pipeline de modelagem.

Essa abordagem evita vazamento de informação do conjunto de teste.

# 11. Remoção de Variáveis Redundantes

In [20]:
df_model = df.drop(
    columns=[
        "DAYS_BIRTH",
        "DAYS_EMPLOYED",
        "DAYS_EMPLOYED_TREATED"
    ]
)

## Justificativa

As informações já estão representadas por:

- AGE_YEARS
- EMPLOYED_YEARS
- EMPLOYED_SPECIAL_VALUE

Mantê-las simultaneamente acrescentaria redundância ao modelo.

# 12. Avaliação das Variáveis de Contato

In [21]:
df_model["FLAG_MOBIL"].value_counts()

,count
FLAG_MOBIL,
1,36457


## Decisão

Remover FLAG_MOBIL.

### Justificativa

A variável apresenta valor constante para todos os registros e não possui capacidade discriminatória.

In [22]:
df_model = df_model.drop(
    columns=["FLAG_MOBIL"]
)

# 13. Dataset Final

In [23]:
print(df_model.shape)

print(
    df_model.isna().sum().sum()
)

print(
    df_model["ID"].duplicated().sum()
)

print(
    df_model["TARGET_BAD"]
    .value_counts()
)

(36457, 20)
0
0
TARGET_BAD
0    35841
1      616
Name: count, dtype: int64


## Resultado Final

A base final:

- não possui valores ausentes;
- não possui IDs duplicados;
- encontra-se pronta para modelagem;
- apresenta forte desbalanceamento da variável alvo, aspecto que será considerado na etapa de modelagem.

In [24]:
df_model.to_csv(
    "base_modelagem_final.csv",
    index=False
)

print(
    "✅ Arquivo 'base_modelagem_final.csv' gerado com sucesso."
)

✅ Arquivo 'base_modelagem_final.csv' gerado com sucesso.
